In [ ]:
import json
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

### Dummy Data Generation for AgriParcel Entity

In [ ]:
def generate_dummy_context_history_with_outliers( entity_id, entity_type, attributes,
                                                 start_time =  datetime(2026, 2, 11, 8, 33, 0),
                                                 n_rows=1000, outlier_ratio=0.1, freq_minutes=1):
    history = []

    outlier_idx = np.random.choice(range(n_rows), size=int(n_rows * outlier_ratio), replace=False)    

    for i in range(n_rows):
        obs_base = start_time + timedelta(minutes=i * freq_minutes) 
        observed_at = (obs_base + timedelta(milliseconds=151)).isoformat(timespec='milliseconds') + "Z"
        time_at = (obs_base + timedelta(milliseconds=152)).isoformat(timespec='milliseconds') + "Z"
        is_outlier = i in outlier_idx
        
        for attr in attributes:
            if is_outlier:
                val = attr["outlier_val"] + np.random.uniform(-1, 1)
            else:
                val = np.random.normal(attr["mean"], attr["std"])
            
            
            if attr["min_val"] is not None:
                val = max(val, attr["min_val"])
            if attr["max_val"] is not None:
                val = min(val, attr["max_val"])
            entry = {
                "observed_at": observed_at,
                "time": time_at,
                "entity_id": entity_id,
                "entity_type": entity_type,
                "attr_name": attr["name"],
                "attr_type": "Property",
                "value": str(round(val, 3)),
                "metadata": "{}"
            }
            history.append(entry)
            
    df = pd.DataFrame(history)
    
    for attr in attributes:
        mask = df["attr_name"] == attr["name"]
        drop_idx = df[mask].sample(frac=attr["null_ratio"]).index
        df.loc[drop_idx, "value"] = np.nan
        df = df.replace({np.nan: None})
    return {"context_history": df.to_dict(orient="records")}, df


In [ ]:
entity_id = "urn:ngsi-ld:AgriParcelRecord:001"
entity_type = "AgriParcelRecord"

attributes = [
    {"name": "leafTemperature", "mean": 13.5, "std": 0.2, "outlier_val": 45.0, "min_val": None, "max_val": None, "null_ratio": 0.05},
    {"name": "leafRelativeHumidity", "mean": 0.74, "std": 0.01, "outlier_val": 0.05, "min_val": 0.0, "max_val": 1.0, "null_ratio": 0.07}
]

n_rows = 1000 # per attribute
start_date = datetime(2026, 2, 11, 8, 33, 0)

context_history, dummy_context_data = generate_dummy_context_history_with_outliers(entity_id, entity_type, attributes,
                                                                                   start_time = start_date,
                                                                                   n_rows=n_rows, outlier_ratio=0.2, 
                                                                                   freq_minutes=10)
#with open("dummy_context_data.json", "w") as f:
#    json.dump(context_history, f, indent=2)
dummy_context_data.to_csv("dummy_context_data.csv", index=False)

In [ ]:
timestamp_columns = ["observed_at", "time"]
timestamp_column = timestamp_columns[1]
dummy_context_data[timestamp_column] = pd.to_datetime(dummy_context_data[timestamp_column])
timestamps = dummy_context_data[timestamp_column].drop_duplicates().sort_values()

print(f"Min: {timestamps.min()}")
print(f"Max: {timestamps.max()}")
print(f"Unique: {len(timestamps)}")

dummy_context_data["value"] = dummy_context_data["value"].astype(float)
print(dummy_context_data.groupby("attr_name")["value"].agg(["count","min", "max", "mean"]))

### Data Drift

In [ ]:
start_date = datetime(2025, 1, 11, 8, 33, 0)


context_history_ref, ref_dummy_context_data = generate_dummy_context_history_with_outliers(entity_id, entity_type, attributes,
                                                                                   start_time = start_date,
                                                                                   n_rows=n_rows, outlier_ratio=0.0, 
                                                                                   freq_minutes=1)

ref_dummy_context_data[timestamp_column] = pd.to_datetime(ref_dummy_context_data[timestamp_column])
ref_dummy_context_data["value"] = ref_dummy_context_data["value"].astype(float)
ref_dummy_context_data.to_csv("dummy_reference_set.csv", index=False)

In [ ]:
def prepare_data_for_drift(df):
    pivoted = df.pivot_table(
        index=['entity_id', 'observed_at'],
        columns='attr_name',
        values='value',
        aggfunc='first'
    ).reset_index()
    
    pivoted = pivoted.drop(columns=['entity_id', 'observed_at'])
    
    return pivoted

ref_prepared = prepare_data_for_drift(ref_dummy_context_data)
current_prepared = prepare_data_for_drift(dummy_context_data)

print(f"Prepared reference shape: {ref_prepared.shape}")
print(f"Prepared current shape: {current_prepared.shape}")
print(f"Columns: {ref_prepared.columns.tolist()}")

#ref_prepared

## Evidently Drift Report

In [ ]:
from evidently.presets import DataDriftPreset
from evidently import Report
from evidently.legacy.pipeline.column_mapping import ColumnMapping

report = Report([DataDriftPreset(method="jensenshannon")])

column_mapping = ColumnMapping()
column_mapping.datetime = "observed_at"

drift_report = report.run(
    reference_data=ref_prepared,
    current_data=current_prepared,
    #column_mapping=column_mapping

)

drift_report

In [ ]:
from scipy.spatial.distance import jensenshannon

def get_evidently_style_js(ref_col, ana_col, bins=20):
    combined_min = min(ref_col.min(), ana_col.min())
    combined_max = max(ref_col.max(), ana_col.max())

    ref_hist, _ = np.histogram(ref_col, bins=bins, range=(combined_min, combined_max))
    ana_hist, _ = np.histogram(ana_col, bins=bins, range=(combined_min, combined_max))

    distance = jensenshannon(ref_hist, ana_hist )
    return distance

js_temp = get_evidently_style_js(ref_prepared['leafRelativeHumidity'], current_prepared['leafRelativeHumidity'], bins=10)
print(f"JS Distance Score: {js_temp}")

In [ ]:
result = json.loads(drift_report.json())
results = []
for metric in result['metrics']:
    if metric['config']['type'] == 'evidently:metric_v2:ValueDrift':
        results.append({
            'column': metric['config']['column'],
            'value': metric['value'],
            'threshold': metric['config']['threshold']
        })

df = pd.DataFrame(results)

df['alert'] = df['value'] > df['threshold']

## NannyML Univariate Drift

In [ ]:
def prepare_data_for_nannyml_drift(df):
    pivoted = df.pivot_table(
        index=['entity_id', 'observed_at'],
        columns='attr_name',
        values='value',
        aggfunc='first'
    ).reset_index()
    
    pivoted.columns.name = None
    pivoted = pivoted.drop(columns='entity_id')
    
    return pivoted

reference_set = prepare_data_for_nannyml_drift(ref_dummy_context_data)
analysis_set = prepare_data_for_nannyml_drift(dummy_context_data)


In [ ]:
import nannyml as nml

column_names = list(set(ref_dummy_context_data['attr_name'].to_list()))

calc = nml.UnivariateDriftCalculator(
    column_names=column_names)

calc.fit(reference_set ,
 #   imestamp_column_name='observed_at',
   # chunk_size=250,
 #   chunk_period="W",
         )

univariate_results = calc.calculate(analysis_set)

for feature_name in column_names:
    figure = univariate_results.plot(kind='drift', column_name=feature_name)
    figure.show()

univariate_results.data


## Distributions

In [ ]:
import pandas as pd
import numpy as np

def calculate_feature_distributions(df, numerical_features=None, categorical_features=None):
    distributions = {}

    if numerical_features is None and categorical_features is None:
        numerical_features = df.select_dtypes(include=[np.number]).columns.tolist()
        categorical_features = df.select_dtypes(include=['object', 'category']).columns.tolist()

    if numerical_features:
        for col in numerical_features:
            if col in df.columns:

                counts, bin_edges = np.histogram(df[col].dropna(), bins=10)

                bin_labels = [f"{bin_edges[i]:.2f}-{bin_edges[i+1]:.2f}" 
                             for i in range(len(bin_edges)-1)]
                
                distributions[col] = {
                    'type': 'numerical',
                    'bins': bin_labels,
                    'counts': counts.tolist(),
                    'bin_edges': bin_edges.tolist()
                }

    if categorical_features:
        for col in categorical_features:
            if col in df.columns:
                value_counts = df[col].value_counts().to_dict()
                
                distributions[col] = {
                    'type': 'categorical',
                    'values': list(value_counts.keys()),
                    'counts': list(value_counts.values())
                }
    
    return distributions


reference_dist = calculate_feature_distributions(
    ref_prepared,
    numerical_features=['leafRelativeHumidity', 'leafTemperature'],
    categorical_features=['category_col'] 
)

current_dist = calculate_feature_distributions(
    current_prepared,
    numerical_features=['leafRelativeHumidity', 'leafTemperature'],
    categorical_features=['category_col']
)


In [ ]:
import matplotlib.pyplot as plt

counts, bin_edges, patches = plt.hist(ref_prepared['leafRelativeHumidity'], bins=10)
print("Counts:", counts)
print("Bin Edges:", bin_edges)
plt.show()